<a href="https://colab.research.google.com/drive/1-syV1GVHq_YyS_7CVfX6d7wMJxANcd9N">Abre este Jupyter en Google Colab</a>

# Creación de Transformadores y Pipelines personalizados

En este notebook se muestra la creación de transformadores y Pipelines personalizados

## Conjunto de datos

### Descripción
NSL-KDD is a data set suggested to solve some of the inherent problems of the KDD'99 data set which are mentioned in. Although, this new version of the KDD data set still suffers from some of the problems discussed by McHugh and may not be a perfect representative of existing real networks, because of the lack of public data sets for network-based IDSs, we believe it still can be applied as an effective benchmark data set to help researchers compare different intrusion detection methods. Furthermore, the number of records in the NSL-KDD train and test sets are reasonable. This advantage makes it affordable to run the experiments on the complete set without the need to randomly select a small portion. Consequently, evaluation results of different research work will be consistent and comparable.

### Ficheros de datos
* <span style="color:green">**KDDTrain+.ARFF**: The full NSL-KDD train set with binary labels in ARFF format</span>
* KDDTrain+.TXT: The full NSL-KDD train set including attack-type labels and difficulty level in CSV format
* KDDTrain+_20Percent.ARFF:	A 20% subset of the KDDTrain+.arff file
* KDDTrain+_20Percent.TXT:	A 20% subset of the KDDTrain+.txt file
* KDDTest+.ARFF:	The full NSL-KDD test set with binary labels in ARFF format
* KDDTest+.TXT:	The full NSL-KDD test set including attack-type labels and difficulty level in CSV format
* KDDTest-21.ARFF:	A subset of the KDDTest+.arff file which does not include records with difficulty level of 21 out of 21
* KDDTest-21.TXT:	A subset of the KDDTest+.txt file which does not include records with difficulty level of 21 out of 21

### Descarga de los ficheros de datos
https://iscxdownloads.cs.unb.ca/iscxdownloads/NSL-KDD/#NSL-KDD

### Referencias adicionales sobre el conjunto de datos
_M. Tavallaee, E. Bagheri, W. Lu, and A. Ghorbani, “A Detailed Analysis of the KDD CUP 99 Data Set,” Submitted to Second IEEE Symposium on Computational Intelligence for Security and Defense Applications (CISDA), 2009._

## Imports

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

## Funciones auxiliares

In [23]:
COLUMN_NAMES = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "class", "difficulty"
]

def load_kdd_dataset(data_path):
    """Lectura del conjunto de datos NSL-KDD desde archivo .txt."""
    return pd.read_csv(data_path, header=None, names=COLUMN_NAMES)

In [24]:
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

## Lectura del conjunto de datos

In [25]:
df = load_kdd_dataset("KDDTrain+.txt")

In [26]:
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20


## División del conjunto de datos

In [27]:
train_set, val_set, test_set = train_val_test_split(df, stratify='protocol_type')

In [28]:
print("Longitud del Training Set:", len(train_set))
print("Longitud del Validation Set:", len(val_set))
print("Longitud del Test Set:", len(test_set))

Longitud del Training Set: 75583
Longitud del Validation Set: 25195
Longitud del Test Set: 25195


## API sklearn

Antes de continuar vamos a hacer una pequeña reseña sobre como funcionan las APIs de sklearn:
* **Estimators**: Cualquier objeto que puede estimar algún parámetro:
    * El propio estimator se forma mediante el método fit(), que siempre toma un dataset como argumento
    * Cualquier otro parámetro de este método, es un hiperparámetro
* **Transformers**: Son estimadores capaces de transformar el conjunto de datos (como Inputer)
    * La transformación se realiza mediante el método transform()
    * Reciben un dataset como parámetro de entrada
* **Predictors**: Son estimadores capaces de realizar predicciónes
    * La predicción se realiza mediante el método predict()
    * Reciben un dataset como entrada
    * Retornan un dataset con las predicciones
    * Tienen un método score() para evaluar el resultado de la predicción

## 1. Construyendo transformadores personalizados

La creación de transformadores propios permite mantener el código mucho más limpio y estructurado a la hora de preparar los datos para los algoritmos de ML. Además, facilitan la reutilización de codigo para otros proyectos. 

Antes de comenzar, vamos a recuperar el conjunto de datos limpio y vamos a separar las etiquetas del resto de los datos, no necesariamente queremos aplicar las mismas transformaciones en ambos conjuntos.

In [29]:
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [30]:
# Para ilustrar esta sección vamos a añadir algunos valores nulos 
# a algunas características del conjunto de datos
X_train.loc[(X_train["src_bytes"]>400) & (X_train["src_bytes"]<800), "src_bytes"] = np.nan
X_train.loc[(X_train["dst_bytes"]>500) & (X_train["dst_bytes"]<2000), "dst_bytes"] = np.nan

In [31]:
X_train

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64559,0,tcp,systat,S0,0.0,0.0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20
67272,0,tcp,http,SF,210.0,NaN,0,0,0,0,...,255,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21
32452,3,tcp,smtp,SF,889.0,328.0,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21
112657,0,tcp,http,SF,284.0,444.0,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21


### Transformadores para atributos numéricos

In [32]:
# Transformador creado para eliminar las filas con valores nulos
class DeleteNanRows(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return X.dropna()

In [33]:
delete_nan = DeleteNanRows()
X_train_prep = delete_nan.fit_transform(X_train)

In [34]:
X_train_prep

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.0,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.0,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.0,0.0,0.0,16
98007,0,udp,domain_u,SF,46.0,139.0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.0,0.0,0.0,20
16447,0,tcp,smtp,SF,1790.0,363.0,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.00,0.0,0.0,0.0,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90665,0,tcp,ftp_data,S0,0.0,0.0,0,0,0,0,...,63,0.25,0.02,0.02,0.00,1.00,1.0,0.0,0.0,18
64559,0,tcp,systat,S0,0.0,0.0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.0,0.0,0.0,20
32452,3,tcp,smtp,SF,889.0,328.0,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.0,0.0,0.0,21
112657,0,tcp,http,SF,284.0,444.0,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,21


In [35]:
# Transofrmador diseñado para escalar de manera sencilla únicamente unas columnas seleccionadas
class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self, attributes):
        self.attributes = attributes
    def fit(self, X, y=None):
        return self  # nothing else to do
    def transform(self, X, y=None):
        X_copy = X.copy()
        scale_attrs = X_copy[self.attributes]
        robust_scaler = RobustScaler()
        X_scaled = robust_scaler.fit_transform(scale_attrs)
        X_scaled = pd.DataFrame(X_scaled, columns=self.attributes, index=X_copy.index)
        for attr in self.attributes:
            X_copy[attr] = X_scaled[attr]
        return X_copy

In [36]:
custom_scaler = CustomScaler(["src_bytes", "dst_bytes"])
X_train_prep = custom_scaler.fit_transform(X_train_prep)

In [37]:
X_train.head(10)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
98007,0,udp,domain_u,SF,46.0,139.0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.00,0.0,0.0,20
16447,0,tcp,smtp,SF,1790.0,363.0,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.00,0.00,0.0,0.0,21
64957,1,tcp,smtp,SF,NaN,329.0,0,0,0,0,...,181,0.65,0.03,0.01,0.01,0.02,0.02,0.0,0.0,21
100052,0,tcp,http,SF,206.0,NaN,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
28800,0,tcp,ftp_data,SF,334.0,0.0,0,0,0,0,...,28,1.00,0.00,1.00,0.11,0.00,0.00,0.0,0.0,11


In [38]:
X_train_prep.head(20)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
31899,0,tcp,private,S0,-0.034632,0.000000,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.0,1.00,0.0,0.0,21
89913,0,tcp,private,S0,-0.034632,0.000000,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.0,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,0.000000,0.000000,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.0,0.00,0.0,0.0,16
98007,0,udp,domain_u,SF,0.164502,0.448387,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.0,0.00,0.0,0.0,20
16447,0,tcp,smtp,SF,7.714286,1.170968,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.0,0.00,0.0,0.0,21
28800,0,tcp,ftp_data,SF,1.411255,0.000000,0,0,0,0,...,28,1.00,0.00,1.00,0.11,0.0,0.00,0.0,0.0,11
78082,0,tcp,ftp_data,SF,44.939394,0.000000,0,0,0,0,...,51,0.23,0.03,0.23,0.04,0.0,0.00,0.0,0.0,21
69315,0,tcp,systat,S0,-0.034632,0.000000,0,0,0,0,...,5,0.02,0.07,0.00,0.00,1.0,1.00,0.0,0.0,20
100360,0,tcp,private,S0,-0.034632,0.000000,0,0,0,0,...,13,0.05,0.07,0.00,0.00,1.0,1.00,0.0,0.0,21
29208,0,tcp,http,SF,1.419913,13.816129,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,21


In [39]:
# Transformador para codificar únicamente las columnas categoricas y devolver un DataFrame
class CustomOneHotEncoding(BaseEstimator, TransformerMixin):
    def __init__(self):
        self._oh = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self._columns = None
    def fit(self, X, y=None):
        X_cat = X.select_dtypes(include=['object'])
        self._columns = pd.get_dummies(X_cat).columns
        self._oh.fit(X_cat)
        return self
    def transform(self, X, y=None):
        X_copy = X.copy()
        X_cat = X_copy.select_dtypes(include=['object'])
        X_num = X_copy.select_dtypes(exclude=['object'])
        X_cat_oh = self._oh.transform(X_cat)
        X_cat_oh = pd.DataFrame(X_cat_oh, 
                                columns=self._columns, 
                                index=X_copy.index)
        X_copy.drop(list(X_cat), axis=1, inplace=True)
        return X_copy.join(X_cat_oh)

## 6. Construyendo Pipelines personalizados

Las pipelines nos permiten agrupar en un flujo de ejecución todas las operaciones de transformación que necesitemos realizar sobre el conjunto de datos, esto facilita muchísimo las transformaciones para diferentes conjuntos de datos. Las cosas a tener en cuenta de estas estructuras son:
* Recibe un conjunto de pares (nombre, estimador)
* Todos menos el último deben ser transformadores
* El pipeline expone los mismos métodos que **el último estimador**
* Cuando se llama al método fit() del pipeline, se llama secuencialmente al método fit_transform() de los estimadores y se les pasa de manera secuencial el output del anterior como input del siguiente. El último invoca el método fit()

In [40]:
# Construcción de un pipeline para los atributos numéricos
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('rbst_scaler', RobustScaler()),
    ])

In [41]:
# La clase imputer no admite valores categoricos, eliminamos los atributos categoricos
X_train_num = X_train.select_dtypes(exclude=['object'])

X_train_prep = num_pipeline.fit_transform(X_train_num)
X_train_prep = pd.DataFrame(X_train_prep, columns=X_train_num.columns, index=X_train_num.index)

In [42]:
X_train_num.head(10)

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,NaN,53508.0,0,0,0,0,0,1,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,0.0,0.0,0,0,0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,304.0,NaN,0,0,0,0,0,1,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,0.0,0.0,0,0,0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,8.0,0.0,0,0,0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
98007,0,46.0,139.0,0,0,0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.00,0.0,0.0,20
16447,0,1790.0,363.0,0,0,0,0,0,1,0,...,137,0.55,0.04,0.01,0.01,0.00,0.00,0.0,0.0,21
64957,1,NaN,329.0,0,0,0,0,0,1,0,...,181,0.65,0.03,0.01,0.01,0.02,0.02,0.0,0.0,21
100052,0,206.0,NaN,0,0,0,0,0,1,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
28800,0,334.0,0.0,0,0,0,0,0,1,0,...,28,1.00,0.00,1.00,0.11,0.00,0.00,0.0,0.0,11


In [43]:
X_train_prep.head(10)

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0.0,0.000000,235.718062,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,1.833333,1.5,0.00,0.00,0.0,0.0,0.333333
31899,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.236735,-0.515789,0.285714,0.000000,0.0,1.00,1.00,0.0,0.0,0.333333
108116,0.0,1.056680,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,0.500000,3.0,0.00,0.00,0.0,0.0,0.333333
89913,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.191837,-0.473684,0.571429,0.000000,0.0,1.00,1.00,0.0,0.0,0.333333
106319,0.0,-0.141700,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.224490,0.515789,-0.428571,16.666667,28.5,0.00,0.00,0.0,0.0,-1.333333
98007,0.0,0.012146,0.612335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.783673,0.515789,-0.285714,0.000000,0.0,0.00,0.00,0.0,0.0,0.000000
16447,0.0,7.072874,1.599119,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.306122,0.042105,0.142857,0.166667,0.5,0.00,0.00,0.0,0.0,0.333333
64957,1.0,0.000000,1.449339,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.485714,0.147368,0.000000,0.166667,0.5,0.02,0.02,0.0,0.0,0.333333
100052,0.0,0.659919,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,0.000000,0.0,0.00,0.00,0.0,0.0,0.333333
28800,0.0,1.178138,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.138776,0.515789,-0.428571,16.666667,5.5,0.00,0.00,0.0,0.0,-3.000000


A continuación se presenta el método ColumnTransformer, que **ejecuta todos los pipelines en paralelo y concatena el resultado** para ello el resultado de los pipelines debe ser un valor numérico

In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_attribs = list(X_train.select_dtypes(exclude=['object']))
cat_attribs = list(X_train.select_dtypes(include=['object']))

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

C:\Users\colmi\AppData\Local\Temp\ipykernel_15720\1734516527.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_attribs = list(X_train.select_dtypes(include=['object']))


In [45]:
X_train_prep = full_pipeline.fit_transform(X_train)

In [46]:
X_train_prep = pd.DataFrame(X_train_prep, columns=list(pd.get_dummies(X_train)), index=X_train.index)

In [47]:
X_train_prep.head(10)

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
113467,0.0,0.000000,235.718062,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
31899,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
108116,0.0,1.056680,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
89913,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
106319,0.0,-0.141700,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
98007,0.0,0.012146,0.612335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
16447,0.0,7.072874,1.599119,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
64957,1.0,0.000000,1.449339,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
100052,0.0,0.659919,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
28800,0.0,1.178138,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [48]:
X_train.head(10)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
98007,0,udp,domain_u,SF,46.0,139.0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.00,0.0,0.0,20
16447,0,tcp,smtp,SF,1790.0,363.0,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.00,0.00,0.0,0.0,21
64957,1,tcp,smtp,SF,NaN,329.0,0,0,0,0,...,181,0.65,0.03,0.01,0.01,0.02,0.02,0.0,0.0,21
100052,0,tcp,http,SF,206.0,NaN,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
28800,0,tcp,ftp_data,SF,334.0,0.0,0,0,0,0,...,28,1.00,0.00,1.00,0.11,0.00,0.00,0.0,0.0,11


## Informe del Notebook

### Descripción General
Este notebook presenta la **creación de transformadores personalizados y pipelines**, permitiendo encapsular y reutilizar flujos complejos de preparación de datos de manera modular y eficiente.

### Dataset Utilizado
- **NSL-KDD**: 125,973 registros con 42 características
- **División**: 60% entrenamiento, 20% validación, 20% pruebas

### Contenido Abarcado
1. **Transformadores personalizados**: DeleteNanRows, CustomScaler, CustomOneHotEncoding
2. **API sklearn**: BaseEstimator, TransformerMixin para crear clases compatibles
3. **Pipelines simples**: Pipeline para encadenar transformaciones secuenciales
4. **ColumnTransformer**: Aplicar transformaciones paralelas a diferentes tipos de características
5. **Reutilización**: Fit en datos completos, transform en subconjuntos

### Cambios Realizados
- Se crearon 3 transformadores personalizados con métodos fit() y transform()
- Se implementó pipeline numérico: Imputer → RobustScaler
- Se construyó ColumnTransformer para procesar features numéricas y categóricas en paralelo
- Se demostró cómo fit en datos completos previene data leakage en test

### Resultados Principales
- **DeleteNanRows**: Eliminó 9,886 filas (13% del dataset)
- **CustomScaler**: Escaló src_bytes y dst_bytes correctamente manteniendo índices
- **Full Pipeline**: Transformó 42 características en 123 características numéricas (39 numéricas + 84 one-hot)
- **Reutilización**: Pipeline se aplicó exitosamente a train, validation y test sets
- Arquitectura escalable lista para producción